# Recolly — Voice pipeline evaluation

```
audio ──[ASR: PhoWhisper]──► transcript ──[Parser: Ollama LLM]──► todo_draft
        Tier 1: WER, KR, RTF        Tier 2: accuracy, F1, deadline
        └──────────── Tier 3: Task Success Rate, Δ_ASR ───────────┘
```

**Run locally** (CPU, quick sanity check): open from `server/notebooks/`, kernel = `server/venv`.

**Run on Kaggle** (GPU, sweep many models):
1. Create a **private** Dataset containing `voice_eval.jsonl` + the `.m4a` files.
2. New Notebook → *File → Import notebook* → upload this file.
3. *Add Input* → your dataset. *Settings* → Accelerator **GPU T4**, **Internet on**.
4. *Run all*. Results land in `/kaggle/working/outputs/`.

⚠️ Quality metrics transfer between machines; **speed metrics (RTF, latency) do not** — Kaggle has a GPU, your server does not.

## 0. Config

In [3]:
import glob
import json
import math
import os
import re
import shutil
import subprocess
import time
import unicodedata
from collections import Counter
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
import requests

IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    found = glob.glob("/kaggle/input/**/voice_eval.jsonl", recursive=True)
    assert found, "Add the private dataset that contains voice_eval.jsonl as notebook Input"
    DATA_DIR = Path(found[0]).parent
    AUDIO_DIR = Path("/kaggle/input")  # searched recursively
    OUTPUT_DIR = Path("/kaggle/working/outputs")
    ASR_MODELS = ["vinai/PhoWhisper-small", "vinai/PhoWhisper-medium", "vinai/PhoWhisper-large"]
    LLM_MODELS = ["qwen2.5:3b", "qwen2.5:7b", "gemma3:4b", "qwen3:4b"]
else:
    DATA_DIR = Path("data").resolve()
    AUDIO_DIR = Path(os.getenv("VOICE_AUDIO_DIR", "~/Downloads/Voice")).expanduser()
    OUTPUT_DIR = Path("outputs").resolve()
    ASR_MODELS = ["vinai/PhoWhisper-small"]
    LLM_MODELS = ["qwen2.5:3b"]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OLLAMA_URL = "http://localhost:11434"
print(f"Kaggle={IS_KAGGLE}\nDATA_DIR={DATA_DIR}\nAUDIO_DIR={AUDIO_DIR}\nASR={ASR_MODELS}\nLLM={LLM_MODELS}")

ModuleNotFoundError: No module named 'pandas'

## 1. Setup: ffmpeg + Ollama (installs only on Kaggle)

In [2]:
def run_shell(command: str) -> None:
    """Run a shell command and stop the notebook if it fails."""
    print("$", command)
    subprocess.run(command, shell=True, check=True)


if IS_KAGGLE:
    if shutil.which("ffmpeg") is None:
        run_shell("apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null")
    if shutil.which("ollama") is None:
        run_shell("apt-get -qq install -y zstd > /dev/null")  # the installer unpacks a .tar.zst
        run_shell("curl -fsSL https://ollama.com/install.sh | sh")

assert shutil.which("ffmpeg"), "ffmpeg is required to decode m4a"
assert shutil.which("ollama"), "Install Ollama first: https://docs.ollama.com"

NameError: name 'IS_KAGGLE' is not defined

In [ ]:
def is_ollama_running() -> bool:
    """Return True when the Ollama HTTP server answers."""
    try:
        return requests.get(f"{OLLAMA_URL}/api/tags", timeout=2).ok
    except requests.RequestException:
        return False


def start_ollama_server() -> None:
    """Start `ollama serve` in the background (if needed) and wait until it is ready."""
    if is_ollama_running():
        print("Ollama already running")
        return
    log_file = open(OUTPUT_DIR / "ollama.log", "w")
    subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=subprocess.STDOUT)
    for _ in range(60):
        if is_ollama_running():
            print("Ollama started")
            return
        time.sleep(1)
    raise RuntimeError(f"Ollama did not start, see {OUTPUT_DIR / 'ollama.log'}")


def pull_ollama_model(model_name: str) -> None:
    """Download an Ollama model (no-op if it is already present)."""
    start = time.perf_counter()
    response = requests.post(f"{OLLAMA_URL}/api/pull", json={"model": model_name, "stream": False}, timeout=3600)
    response.raise_for_status()
    print(f"pulled {model_name} in {time.perf_counter() - start:.0f}s")


start_ollama_server()
for llm_model in LLM_MODELS:
    pull_ollama_model(llm_model)

## 2. Data

One line of `voice_eval.jsonl` = one example. `audio: null` → text-only example (only the parser is evaluated).
`tags` let you slice results (e.g. `relative_date`, `no_deadline`).

In [ ]:
def load_eval_examples(path: Path) -> list[dict]:
    """Read the JSONL eval set, skipping blank lines."""
    with open(path, encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


def find_audio_file(file_name: str) -> Path | None:
    """Find an audio file by name anywhere under AUDIO_DIR."""
    matches = list(AUDIO_DIR.rglob(file_name))
    return matches[0] if matches else None


examples = load_eval_examples(DATA_DIR / "voice_eval.jsonl")
for example in examples:
    if example["audio"]:
        assert find_audio_file(example["audio"]), f"Missing audio file: {example['audio']}"

print(f"{len(examples)} examples, {sum(1 for e in examples if e['audio'])} with audio")
pd.DataFrame([{"id": e["id"], "audio": e["audio"], "text": e["transcript_gold"], "tags": ", ".join(e["tags"])}
              for e in examples])

## 3. Text normalization — why `"ộ" != "ộ"`

Vietnamese diacritics can be stored *composed* (NFC, 1 code point) or *decomposed* (NFD, base letter + combining marks).
They look identical but compare as different strings → WER would count them as errors. Always normalize to NFC first.

In [ ]:
def normalize_vietnamese_text(text: str) -> str:
    """NFC-normalize, lowercase and strip punctuation so transcripts can be compared fairly."""
    text = unicodedata.normalize("NFC", text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(text.split())


composed = "ộ"
decomposed = unicodedata.normalize("NFD", composed)
print("equal before normalizing:", composed == decomposed, "| code points:", len(composed), "vs", len(decomposed))
print("equal after normalizing: ", normalize_vietnamese_text(composed) == normalize_vietnamese_text(decomposed))
print(normalize_vietnamese_text("Ngày mai, 9 GIỜ nộp báo cáo!"))

## 4. Metrics — ✍️ YOUR TURN

Each function = one formula. Replace `raise NotImplementedError` with your implementation, then run the test cell below.
The pipeline (sections 5–7) still runs while metrics are TODO; their columns just show `TODO`.

Suggested order: `accuracy` → `real_time_factor` → `token_f1` → `keyword_recall` → `macro_f1` → `mean_absolute_error_hours` → `confidence_interval_95` → `word_error_rate` (the hardest, worth it).

In [ ]:
def safe_divide(numerator: float, denominator: float) -> float:
    """Return numerator / denominator, defining 0/0 (or x/0) as 0."""
    return numerator / denominator if denominator else 0.0


def accuracy(predictions: list, labels: list) -> float:
    """Fraction of exact matches.

    accuracy = (1/n) * Σ_i 1[ŷ_i == y_i]
    """
    correct_count = sum(1 for predicted, label in zip(predictions, labels) if predicted == label)
    return safe_divide(correct_count, len(labels))


def f1_from_precision_recall(precision: float, recall: float) -> float:
    """Harmonic mean of precision and recall.

    F1 = 2·P·R / (P + R)   (define 0/0 = 0)
    """
    return safe_divide(2 * precision * recall, precision + recall)


def macro_f1(predictions: list, labels: list) -> float:
    """Unweighted mean of per-class F1 over every class seen in predictions or labels.

    P_c = TP_c / (TP_c + FP_c),  R_c = TP_c / (TP_c + FN_c)   (define 0/0 = 0)
    F1_c = 2·P_c·R_c / (P_c + R_c)
    macroF1 = (1/|C|) * Σ_c F1_c
    """
    pairs = list(zip(predictions, labels))
    classes = set(predictions) | set(labels)
    per_class_f1 = []
    for current_class in classes:
        true_positive = sum(1 for predicted, label in pairs if predicted == current_class and label == current_class)
        false_positive = sum(1 for predicted, label in pairs if predicted == current_class and label != current_class)
        false_negative = sum(1 for predicted, label in pairs if predicted != current_class and label == current_class)
        precision = safe_divide(true_positive, true_positive + false_positive)
        recall = safe_divide(true_positive, true_positive + false_negative)
        per_class_f1.append(f1_from_precision_recall(precision, recall))
    return safe_divide(sum(per_class_f1), len(per_class_f1))


def token_f1(predicted_text: str, reference_text: str) -> float:
    """Bag-of-tokens F1 between two normalized texts (tokens = Vietnamese syllables).

    overlap = |Counter(pred) ∩ Counter(ref)|
    P = overlap / |pred|,  R = overlap / |ref|,  F1 = 2PR / (P + R)
    """
    predicted_tokens = normalize_vietnamese_text(predicted_text).split()
    reference_tokens = normalize_vietnamese_text(reference_text).split()
    overlap = sum((Counter(predicted_tokens) & Counter(reference_tokens)).values())
    precision = safe_divide(overlap, len(predicted_tokens))
    recall = safe_divide(overlap, len(reference_tokens))
    return f1_from_precision_recall(precision, recall)


def word_error_rate(hypothesis: str, reference: str) -> float:
    """Syllable-level WER via Levenshtein distance.

    WER = (S + D + I) / N,  N = number of reference syllables
    d[i][j] = edit distance between ref[:i] and hyp[:j]
    d[i][j] = min(d[i-1][j] + 1,                         # deletion
                  d[i][j-1] + 1,                         # insertion
                  d[i-1][j-1] + (ref[i-1] != hyp[j-1]))  # substitution / match
    """
    reference_tokens = normalize_vietnamese_text(reference).split()
    hypothesis_tokens = normalize_vietnamese_text(hypothesis).split()
    rows, columns = len(reference_tokens) + 1, len(hypothesis_tokens) + 1

    # d[i][0] = i (delete all i reference tokens), d[0][j] = j (insert all j hypothesis tokens)
    distance = [[0] * columns for _ in range(rows)]
    for i in range(rows):
        distance[i][0] = i
    for j in range(columns):
        distance[0][j] = j

    for i in range(1, rows):
        for j in range(1, columns):
            substitution_cost = int(reference_tokens[i - 1] != hypothesis_tokens[j - 1])
            distance[i][j] = min(distance[i - 1][j] + 1,
                                 distance[i][j - 1] + 1,
                                 distance[i - 1][j - 1] + substitution_cost)

    return safe_divide(distance[-1][-1], len(reference_tokens))


def keyword_recall(transcript: str, keywords: list[str]) -> float:
    """Fraction of important keywords that appear in the transcript (after normalization).

    KR = #keywords found / #keywords
    Matches whole syllables only, so "thi" does not match inside "thích".
    """
    padded_transcript = f" {normalize_vietnamese_text(transcript)} "
    found_count = sum(1 for keyword in keywords if f" {normalize_vietnamese_text(keyword)} " in padded_transcript)
    return safe_divide(found_count, len(keywords))


def real_time_factor(processing_seconds: float, audio_seconds: float) -> float:
    """How many seconds of compute per second of audio (< 1 means faster than real time).

    RTF = t_processing / t_audio
    """
    return safe_divide(processing_seconds, audio_seconds)


def mean_absolute_error_hours(predicted_times: list[datetime], true_times: list[datetime]) -> float:
    """Average absolute deadline error in hours.

    MAE_h = (1/m) * Σ_i |t̂_i − t_i| / 3600s
    """
    total_hours = sum(abs((predicted - true).total_seconds()) / 3600
                      for predicted, true in zip(predicted_times, true_times))
    return safe_divide(total_hours, len(true_times))


def confidence_interval_95(proportion: float, sample_size: int) -> tuple[float, float]:
    """Normal-approximation 95% confidence interval for a proportion.

    p ± 1.96 · sqrt(p(1 − p) / n)
    """
    margin = 1.96 * math.sqrt(proportion * (1 - proportion) / sample_size)
    return proportion - margin, proportion + margin

In [ ]:
def check_metric(name: str, test) -> None:
    """Run one metric test and print ✅ / ❌ / ⬜ TODO."""
    try:
        test()
        print(f"✅ {name}")
    except NotImplementedError:
        print(f"⬜ {name}: TODO")
    except AssertionError as error:
        print(f"❌ {name}: {error}")


def close(a, b) -> bool:
    """Float comparison with a small tolerance."""
    return math.isclose(a, b, abs_tol=1e-4)


def test_accuracy():
    assert close(accuracy(["a", "b", "c"], ["a", "b", "d"]), 2 / 3), "expected 2/3"

def test_macro_f1():
    assert close(macro_f1(["a", "a", "b"], ["a", "b", "b"]), 2 / 3), "expected 2/3 (both classes F1 = 2/3)"
    assert close(macro_f1(["a", "a"], ["a", "a"]), 1.0), "perfect predictions → 1.0"

def test_token_f1():
    assert close(token_f1("Nộp bài tập lớn", "nộp bài tập lớn cấu trúc dữ liệu"), 2 / 3), "P=1, R=4/8 → 2/3"
    assert close(token_f1("abc", "xyz"), 0.0), "no overlap → 0"

def test_word_error_rate():
    assert close(word_error_rate("ngày mai chín giờ gian nộp", "ngày mai chín giờ sáng nộp"), 1 / 6), "1 substitution / 6"
    assert close(word_error_rate("a b", "a b c"), 1 / 3), "1 deletion / 3"
    assert close(word_error_rate("a x b c", "a b c"), 1 / 3), "1 insertion / 3"
    assert close(word_error_rate("Ngày MAI!", "ngày mai"), 0.0), "normalize first"

def test_keyword_recall():
    assert close(keyword_recall("tối mai 8 giờ ôn thi giải tích", ["tối mai", "thi", "Giải Tích", "vật lý"]), 3 / 4)

def test_real_time_factor():
    assert close(real_time_factor(14, 4), 3.5), "the CPU demo: 14s for 4s of audio"

def test_mean_absolute_error_hours():
    t = datetime(2026, 11, 10, 8, 0)
    assert close(mean_absolute_error_hours([t, t], [t + timedelta(hours=2), t]), 1.0)

def test_confidence_interval_95():
    low, high = confidence_interval_95(0.8, 50)
    assert close(low, 0.8 - 0.110873) and close(high, 0.8 + 0.110873), "0.8 ± 0.1109"


for metric_test in [test_accuracy, test_real_time_factor, test_token_f1, test_keyword_recall, test_macro_f1,
                    test_mean_absolute_error_hours, test_confidence_interval_95, test_word_error_rate]:
    check_metric(metric_test.__name__.removeprefix("test_"), metric_test)

## 5. Run ASR (PhoWhisper)

Results are cached in `outputs/asr_results.jsonl`, so re-running skips work already done.

### The whole pipeline at a glance

```
                        ┌──────────── optional (section 5b) ────────────┐
                        │  + noise (white / pink / babble / real file)  │
                        │    mixed = signal + α·noise   at SNR 20…0 dB  │
                        └───────────────────────┬───────────────────────┘
 School.m4a                                     ▼
    │ ① ffmpeg decode → 16 kHz mono               waveform (75093,)        ← 4.69 s × 16000 samples
    ▼                                             │
 ┌──────────────── self.pipe(...) = transformers pipeline ────────────────────────────────────┐
 │  ② feature extractor   STFT 25 ms / 10 ms → 80 Mel filters → log   log-Mel (1, 80, 3000)  │  fixed formula
 │  ③ ENCODER  (Transformer "ears")                                    states  (1, 1500, 768) │  ┐ PhoWhisper
 │  ④ DECODER  (Transformer "writer"), one token at a time,                                   │  │ = the only
 │             primed with <|vi|><|transcribe|>                         token ids (22 tokens)  │  ┘ trained part
 │  ⑤ tokenizer: ids → UTF-8 bytes → text                                                     │  lookup table
 └────────────────────────────────────────────────────────────────────────────────────────────┘
    ▼
 transcript  "cặc lịch mời học trong trường lúc sáu giờ sáng."
    │                                             ── Tier 1 (ASR):    WER · Keyword Recall · RTF
    ▼
 Ollama LLM + JSON schema  (section 6)
    ▼
 todo_draft  {title, category, priority, deadline_at}
                                                  ── Tier 2 (parser): accuracy · macro-F1 · deadline MAE · title token-F1
                                                  ── Tier 3 (end-to-end): Task Success Rate · Δ_ASR = TSR(gold) − TSR(ASR)
```

| step | object in code | what it does |
|---|---|---|
| ① | `load_waveform()` (5b) / pipeline's own ffmpeg (5) | file → numbers in [-1, 1] |
| ② | `pipe.feature_extractor` | waveform → log-Mel spectrogram |
| ③ ④ | `pipe.model` | log-Mel → token ids (this is what "PhoWhisper" really is) |
| ⑤ | `pipe.tokenizer` | token ids → text |

`AsrRunner.transcribe(path)` runs ①–⑤; `AsrRunner.transcribe_waveform(array)` starts at ② so we can add noise to the waveform first.

In [ ]:
import torch
import transformers
from transformers import pipeline


def audio_duration_seconds(audio_path: Path) -> float:
    """Read audio length with ffprobe."""
    output = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(audio_path)],
        capture_output=True, text=True, check=True)
    return float(output.stdout.strip())


def load_jsonl_cache(path: Path) -> list[dict]:
    """Read cached result rows (empty list if the file does not exist yet)."""
    return load_eval_examples(path) if path.exists() else []


def append_jsonl(path: Path, row: dict) -> None:
    """Append one result row to a JSONL cache file."""
    with open(path, "a", encoding="utf-8") as file:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")


ASR_SAMPLE_RATE = 16_000  # Whisper models expect 16 kHz mono audio


class AsrRunner:
    """Holds one PhoWhisper model and transcribes audio files with it."""

    def __init__(self, model_name: str):
        """Load the model on GPU (fp16) when available, otherwise CPU (fp32)."""
        self.model_name = model_name
        use_gpu = torch.cuda.is_available()
        dtype = torch.float16 if use_gpu else torch.float32
        major_version = int(transformers.__version__.split(".")[0])
        dtype_argument = {"dtype": dtype} if major_version >= 5 else {"torch_dtype": dtype}
        self.pipe = pipeline("automatic-speech-recognition", model=model_name,
                             device=0 if use_gpu else "cpu", chunk_length_s=30, **dtype_argument)

    def transcribe(self, audio_path: Path) -> tuple[str, float]:
        """Return (text, processing seconds) for one audio file, forcing Vietnamese."""
        start = time.perf_counter()
        output = self.pipe(str(audio_path), generate_kwargs={"language": "vi", "task": "transcribe"})
        return output["text"].strip(), time.perf_counter() - start

    def transcribe_waveform(self, waveform) -> tuple[str, float]:
        """Return (text, processing seconds) for a 16 kHz mono waveform already in memory (e.g. noisy mix)."""
        start = time.perf_counter()
        output = self.pipe({"raw": waveform, "sampling_rate": ASR_SAMPLE_RATE},
                           generate_kwargs={"language": "vi", "task": "transcribe"})
        return output["text"].strip(), time.perf_counter() - start


ASR_CACHE = OUTPUT_DIR / "asr_results.jsonl"
audio_examples = [e for e in examples if e["audio"]]

for asr_model in ASR_MODELS:
    done_ids = {r["id"] for r in load_jsonl_cache(ASR_CACHE) if r["asr_model"] == asr_model}
    todo = [e for e in audio_examples if e["id"] not in done_ids]
    if not todo:
        print(f"{asr_model}: cached")
        continue
    runner = AsrRunner(asr_model)
    runner.transcribe(find_audio_file(todo[0]["audio"]))  # warm-up, not timed
    for example in todo:
        audio_path = find_audio_file(example["audio"])
        text, seconds = runner.transcribe(audio_path)
        append_jsonl(ASR_CACHE, {"asr_model": asr_model, "id": example["id"], "transcript": text,
                                 "processing_seconds": round(seconds, 3),
                                 "audio_seconds": round(audio_duration_seconds(audio_path), 3)})
        print(f"{asr_model} | {example['id']} | {text}")
    del runner
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

asr_results = load_jsonl_cache(ASR_CACHE)

## 5b. Noise robustness — how much does background noise hurt PhoWhisper?

We take every **clean** recording and add noise at a fixed **SNR** (signal-to-noise ratio), then re-run ASR.
Same sentence, same speaker, only the noise changes → any rise in WER is caused by the noise.

```
SNR_dB = 10 · log10(P_signal / P_noise)          P = (1/N) · Σ x_i²
α      = sqrt(P_signal / (P_noise · 10^(SNR_dB/10)))
mixed  = signal + α · noise
```

| noise type | built from | simulates |
|---|---|---|
| `white`  | Gaussian random samples (flat spectrum) | hiss, fan, cheap mic |
| `pink`   | white noise shaped to power ∝ 1/f | traffic, rain, engine rumble |
| `babble` | sum of **other clips in this eval set** (different sentences) | canteen, café, crowded corridor |
| `file:<name>` | any real ambient recording put in `VOICE_NOISE_DIR` (optional) | the real place you recorded |

SNR levels: **20 dB** (slightly noisy) → 10 → 5 → **0 dB** (noise as loud as the voice).
A few mixed clips are saved to `outputs/noisy_samples/` — **listen to them** to get a feel for what each SNR sounds like.

In [ ]:
import wave
import zlib

import numpy as np

SNR_LEVELS_DB = [20, 10, 5, 0]
NOISE_TYPES = ["white", "pink", "babble"]
BABBLE_TALKERS = 4  # how many other clips are summed into one babble track
NOISY_ASR_MODELS = ASR_MODELS  # shrink this list on Kaggle if the sweep takes too long
NOISE_DIR = Path(os.getenv("VOICE_NOISE_DIR", "~/Downloads/Voice/noise")).expanduser()
NOISY_ASR_CACHE = OUTPUT_DIR / "asr_noisy_results.jsonl"
NOISY_SAMPLES_DIR = OUTPUT_DIR / "noisy_samples"


def load_waveform(audio_path: Path) -> np.ndarray:
    """Decode any audio file to a 16 kHz mono float32 waveform in [-1, 1] using ffmpeg."""
    output = subprocess.run(
        ["ffmpeg", "-v", "error", "-i", str(audio_path), "-f", "f32le", "-ac", "1", "-ar", str(ASR_SAMPLE_RATE), "-"],
        capture_output=True, check=True)
    return np.frombuffer(output.stdout, dtype=np.float32).copy()


def save_waveform(waveform: np.ndarray, path: Path) -> None:
    """Write a float waveform in [-1, 1] as a 16-bit PCM WAV file (so you can listen to it)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    pcm = (np.clip(waveform, -1, 1) * 32767).astype(np.int16)
    with wave.open(str(path), "wb") as file:
        file.setnchannels(1)
        file.setsampwidth(2)
        file.setframerate(ASR_SAMPLE_RATE)
        file.writeframes(pcm.tobytes())


def signal_power(waveform: np.ndarray) -> float:
    """Average power of a waveform.

    P = (1/N) * Σ_i x_i²
    """
    return float(np.mean(waveform.astype(np.float64) ** 2))


def measured_snr_db(signal: np.ndarray, noise: np.ndarray) -> float:
    """Signal-to-noise ratio in decibels.

    SNR_dB = 10 * log10(P_signal / P_noise)
    """
    return 10 * math.log10(signal_power(signal) / signal_power(noise))


def scale_noise_to_snr(signal: np.ndarray, noise: np.ndarray, snr_db: float) -> np.ndarray:
    """Rescale noise so that signal vs. scaled noise has exactly the target SNR.

    α = sqrt(P_signal / (P_noise * 10^(SNR_dB / 10))),   returns α * noise
    """
    alpha = math.sqrt(signal_power(signal) / (signal_power(noise) * 10 ** (snr_db / 10)))
    return alpha * noise


def mix_at_snr(signal: np.ndarray, noise: np.ndarray, snr_db: float) -> np.ndarray:
    """Add noise to a signal at the target SNR, then peak-normalize if the sum would clip.

    mixed = signal + α * noise;  if max|mixed| > 1: mixed = mixed / max|mixed|  (SNR unchanged)
    """
    mixed = signal + scale_noise_to_snr(signal, noise, snr_db)
    peak = float(np.max(np.abs(mixed)))
    return (mixed / peak if peak > 1 else mixed).astype(np.float32)


class NoiseGenerator:
    """Builds reproducible noise tracks (white, pink, babble, real files) of any length."""

    def __init__(self, clean_clips: dict[str, np.ndarray], gold_texts: dict[str, str], noise_files: list[Path]):
        """Keep the clean clips (babble source) and the optional real-noise recordings."""
        self.clean_clips = clean_clips
        self.gold_texts = gold_texts
        self.noise_waveforms = {f"file:{path.stem}": load_waveform(path) for path in noise_files}

    def available_noise_types(self) -> list[str]:
        """Synthetic types plus one type per real noise file."""
        return NOISE_TYPES + sorted(self.noise_waveforms)

    def make_noise(self, noise_type: str, length: int, example_id: str) -> np.ndarray | None:
        """Return a noise track of `length` samples, or None if this type cannot be built."""
        rng = np.random.default_rng(zlib.crc32(f"{example_id}|{noise_type}".encode()))  # same noise every run
        if noise_type == "white":
            return rng.standard_normal(length)
        if noise_type == "pink":
            return self._pink_noise(length, rng)
        if noise_type == "babble":
            return self._babble_noise(length, rng, example_id)
        return self._fit_length(self.noise_waveforms[noise_type], length, rng)

    @staticmethod
    def _pink_noise(length: int, rng: np.random.Generator) -> np.ndarray:
        """Pink noise: shape white noise in the frequency domain.

        |X(f)|² ∝ 1/f   ⇒   multiply each FFT bin of white noise by 1/sqrt(f)
        """
        spectrum = np.fft.rfft(rng.standard_normal(length))
        frequencies = np.fft.rfftfreq(length)
        frequencies[0] = frequencies[1]  # avoid dividing by zero at DC
        return np.fft.irfft(spectrum / np.sqrt(frequencies), n=length)

    def _babble_noise(self, length: int, rng: np.random.Generator, example_id: str) -> np.ndarray | None:
        """Crowd noise: sum of other clips (different sentence), each scaled to equal power.

        babble = Σ_k clip_k / sqrt(P_k)
        """
        own_text = self.gold_texts[example_id]
        candidates = [clip_id for clip_id in self.clean_clips
                      if clip_id != example_id and self.gold_texts[clip_id] != own_text]
        if not candidates:
            return None
        chosen = rng.choice(candidates, size=min(BABBLE_TALKERS, len(candidates)), replace=False)
        babble = np.zeros(length)
        for clip_id in chosen:
            clip = self._fit_length(self.clean_clips[clip_id], length, rng)
            babble += clip / math.sqrt(signal_power(clip))
        return babble

    @staticmethod
    def _fit_length(waveform: np.ndarray, length: int, rng: np.random.Generator) -> np.ndarray:
        """Loop a waveform until it is long enough, then cut a random window of `length` samples."""
        repeats = math.ceil((length + len(waveform)) / len(waveform))
        looped = np.tile(waveform, repeats)
        start = int(rng.integers(0, len(waveform)))
        return looped[start:start + length].astype(np.float64)


# --- sanity check: the mixer must hit the target SNR exactly ---
test_rng = np.random.default_rng(0)
test_signal = np.sin(np.linspace(0, 200 * np.pi, ASR_SAMPLE_RATE)) * 0.3
for target_snr in SNR_LEVELS_DB:
    scaled = scale_noise_to_snr(test_signal, test_rng.standard_normal(ASR_SAMPLE_RATE), target_snr)
    assert math.isclose(measured_snr_db(test_signal, scaled), target_snr, abs_tol=1e-6)
print("✅ scale_noise_to_snr hits the target SNR")

# --- build noise + run ASR on every (clip, noise type, SNR) ---
clean_clips = {e["id"]: load_waveform(find_audio_file(e["audio"])) for e in audio_examples}
noise_files = sorted(p for p in NOISE_DIR.glob("*") if p.suffix.lower() in {".wav", ".m4a", ".mp3", ".ogg", ".flac"}) \
    if NOISE_DIR.exists() else []
noise_generator = NoiseGenerator(clean_clips, {e["id"]: e["transcript_gold"] for e in examples}, noise_files)
print("noise types:", noise_generator.available_noise_types())

for asr_model in NOISY_ASR_MODELS:
    done = {(r["id"], r["noise_type"], r["snr_db"]) for r in load_jsonl_cache(NOISY_ASR_CACHE)
            if r["asr_model"] == asr_model}
    todo = [(e, noise_type, snr) for e in audio_examples for noise_type in noise_generator.available_noise_types()
            for snr in SNR_LEVELS_DB if (e["id"], noise_type, snr) not in done]
    if not todo:
        print(f"{asr_model}: cached")
        continue
    runner = AsrRunner(asr_model)
    runner.transcribe_waveform(clean_clips[todo[0][0]["id"]])  # warm-up, not timed
    for example, noise_type, snr in todo:
        clean = clean_clips[example["id"]]
        noise = noise_generator.make_noise(noise_type, len(clean), example["id"])
        if noise is None:
            continue  # e.g. babble needs at least one other clip
        mixed = mix_at_snr(clean, noise, snr)
        if example["id"] == audio_examples[0]["id"]:
            save_waveform(mixed, NOISY_SAMPLES_DIR / f"{example['id']}_{noise_type.replace(':', '-')}_{snr}dB.wav")
        text, seconds = runner.transcribe_waveform(mixed)
        append_jsonl(NOISY_ASR_CACHE, {"asr_model": asr_model, "id": example["id"], "noise_type": noise_type,
                                       "snr_db": snr, "transcript": text, "processing_seconds": round(seconds, 3)})
        print(f"{asr_model} | {example['id']} | {noise_type:7} | {snr:>2} dB | {text}")
    del runner
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

noisy_asr_results = load_jsonl_cache(NOISY_ASR_CACHE)

# --- report: WER / keyword recall per noise type and SNR (clean = baseline from section 5) ---
gold_by_id = {e["id"]: e for e in examples}
noise_rows = [{"asr_model": r["asr_model"], "noise_type": r["noise_type"], "snr_db": r["snr_db"],
               "WER": word_error_rate(r["transcript"], gold_by_id[r["id"]]["transcript_gold"]),
               "KR": keyword_recall(r["transcript"], gold_by_id[r["id"]]["keywords"])}
              for r in noisy_asr_results]
clean_rows = [{"asr_model": r["asr_model"], "snr_db": "clean",
               "WER": word_error_rate(r["transcript"], gold_by_id[r["id"]]["transcript_gold"]),
               "KR": keyword_recall(r["transcript"], gold_by_id[r["id"]]["keywords"])}
              for r in asr_results]
if noise_rows:
    noise_table = pd.DataFrame(noise_rows)
    clean_table = pd.DataFrame(clean_rows).groupby("asr_model")[["WER", "KR"]].mean()
    column_order = ["clean"] + SNR_LEVELS_DB
    for metric in ["WER", "KR"]:
        pivot = noise_table.pivot_table(index=["asr_model", "noise_type"], columns="snr_db", values=metric, aggfunc="mean")
        pivot["clean"] = [clean_table.loc[model, metric] for model, _ in pivot.index]
        print(f"{metric} by noise type × SNR (dB)   — lower WER / higher KR is better")
        display(pivot[[c for c in column_order if c in pivot.columns]].round(3))

## 6. Run parser (Ollama + JSON schema)

**Baseline**: the LLM computes `deadline_at` itself. Later experiment: let the LLM output *relative* time
(`weekday`, `day_offset`, `time`) and compute the date in Python — then compare the numbers.

Each LLM is run on two inputs: the **gold transcript** and every **ASR transcript** → lets us compute Δ_ASR.

In [ ]:
VIETNAMESE_WEEKDAYS = ["Thứ Hai", "Thứ Ba", "Thứ Tư", "Thứ Năm", "Thứ Sáu", "Thứ Bảy", "Chủ Nhật"]

TODO_DRAFT_SCHEMA = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "category": {"type": "string", "enum": ["study", "assignment", "exam", "personal", "other"]},
        "priority": {"type": "string", "enum": ["low", "medium", "high"]},
        "deadline_at": {"type": ["string", "null"], "description": "ISO 8601 with +07:00 offset, or null"},
    },
    "required": ["title", "category", "priority", "deadline_at"],
}


def describe_reference_time(reference_time: str) -> str:
    """Turn an ISO timestamp into 'Thứ Ba, 2026-11-10 14:00 (+07:00)' for the prompt."""
    moment = datetime.fromisoformat(reference_time)
    return f"{VIETNAMESE_WEEKDAYS[moment.weekday()]}, {moment:%Y-%m-%d %H:%M} (+07:00)"


def build_system_prompt(reference_time: str) -> str:
    """System prompt telling the LLM how to turn one utterance into a todo."""
    return (
        "Bạn trích xuất MỘT việc cần làm từ câu nói tiếng Việt của sinh viên.\n"
        f"Thời điểm hiện tại: {describe_reference_time(reference_time)}.\n"
        "- title: ngắn gọn, viết hoa chữ đầu, không chứa thời gian.\n"
        "- category: study (học/ôn), assignment (bài tập/nộp/báo cáo), exam (thi/kiểm tra), personal, other.\n"
        "- priority: high nếu có 'gấp', 'quan trọng'; low nếu việc vặt; còn lại medium.\n"
        "- deadline_at: ISO 8601 +07:00. Có ngày mà không có giờ thì dùng 23:59. Không nhắc thời gian thì null."
    )


class OllamaTodoParser:
    """Turns a transcript into a todo_draft dict using an Ollama model with structured output."""

    def __init__(self, model_name: str):
        """Remember which Ollama model to call."""
        self.model_name = model_name

    def parse(self, transcript: str, reference_time: str) -> tuple[dict | None, float, str | None]:
        """Return (draft or None, seconds, error message or None)."""
        payload = {
            "model": self.model_name,
            "stream": False,
            "format": TODO_DRAFT_SCHEMA,
            "options": {"temperature": 0},
            "messages": [{"role": "system", "content": build_system_prompt(reference_time)},
                         {"role": "user", "content": transcript}],
        }
        if self.model_name.startswith("qwen3"):
            payload["think"] = False  # thinking mode is slow and not needed here
        start = time.perf_counter()
        try:
            response = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=600)
            response.raise_for_status()
            draft = json.loads(response.json()["message"]["content"])
            return draft, time.perf_counter() - start, None
        except (requests.RequestException, json.JSONDecodeError, KeyError) as error:
            return None, time.perf_counter() - start, str(error)


def parser_inputs() -> list[tuple[str, dict, str]]:
    """List (source, example, transcript): gold text for all examples + every ASR output."""
    by_id = {e["id"]: e for e in examples}
    inputs = [("gold", e, e["transcript_gold"]) for e in examples]
    inputs += [(f"asr:{r['asr_model']}", by_id[r["id"]], r["transcript"]) for r in asr_results]
    return inputs


PARSER_CACHE = OUTPUT_DIR / "parser_results.jsonl"

for llm_model in LLM_MODELS:
    done = {(r["source"], r["id"]) for r in load_jsonl_cache(PARSER_CACHE) if r["llm_model"] == llm_model}
    todo = [(s, e, t) for s, e, t in parser_inputs() if (s, e["id"]) not in done]
    if not todo:
        print(f"{llm_model}: cached")
        continue
    parser = OllamaTodoParser(llm_model)
    parser.parse("test", examples[0]["reference_time"])  # warm-up: loads the model into memory
    for source, example, transcript in todo:
        draft, seconds, error = parser.parse(transcript, example["reference_time"])
        append_jsonl(PARSER_CACHE, {"llm_model": llm_model, "source": source, "id": example["id"],
                                    "transcript": transcript, "draft": draft,
                                    "processing_seconds": round(seconds, 3), "error": error})
        print(f"{llm_model} | {source:28} | {example['id']} | {json.dumps(draft, ensure_ascii=False)}")

parser_results = load_jsonl_cache(PARSER_CACHE)

## 7. Report

In [ ]:
def safe_metric(function, *args):
    """Call a metric, returning 'TODO' while it is not implemented yet."""
    try:
        return round(function(*args), 3)
    except NotImplementedError:
        return "TODO"


VIETNAM_TIMEZONE = timezone(timedelta(hours=7))


def parse_deadline(value: str | None) -> datetime | None:
    """Parse an ISO 8601 deadline; None when missing or malformed (LLMs emit things like '+07:0:00').

    A deadline without an offset is assumed to be Vietnam time (+07:00), so it can be subtracted safely.
    """
    if not isinstance(value, str) or not value:
        return None
    try:
        moment = datetime.fromisoformat(value)
    except ValueError:
        return None
    return moment if moment.tzinfo else moment.replace(tzinfo=VIETNAM_TIMEZONE)


def is_invalid_deadline(value) -> bool:
    """True when the model returned a deadline string that cannot be parsed."""
    return value is not None and parse_deadline(value) is None


def same_deadline_day(predicted: str | None, expected: str | None) -> bool:
    """True when both deadlines are null, or both fall on the same calendar day (malformed = wrong)."""
    if predicted is None or expected is None:
        return predicted is None and expected is None
    predicted_moment, expected_moment = parse_deadline(predicted), parse_deadline(expected)
    if predicted_moment is None or expected_moment is None:
        return False
    return predicted_moment.date() == expected_moment.date()


def is_task_success(draft: dict | None, expected: dict) -> bool:
    """Headline metric: category right AND deadline on the right day AND title roughly right."""
    if draft is None:
        return False
    try:
        title_ok = token_f1(draft.get("title", ""), expected["title"]) >= 0.5
    except NotImplementedError:
        title_ok = True  # until token_f1 exists, judge only category + deadline
    return (draft.get("category") == expected["category"]
            and same_deadline_day(draft.get("deadline_at"), expected["deadline_at"])
            and title_ok)


by_id = {e["id"]: e for e in examples}

# --- Tier 1: ASR ---
asr_rows = []
for asr_model in ASR_MODELS:
    rows = [r for r in asr_results if r["asr_model"] == asr_model]
    if not rows:
        continue
    asr_rows.append({
        "asr_model": asr_model, "n": len(rows),
        "WER": safe_metric(lambda: sum(word_error_rate(r["transcript"], by_id[r["id"]]["transcript_gold"])
                                       for r in rows) / len(rows)),
        "keyword_recall": safe_metric(lambda: sum(keyword_recall(r["transcript"], by_id[r["id"]]["keywords"])
                                                  for r in rows) / len(rows)),
        "RTF (this machine)": safe_metric(lambda: sum(real_time_factor(r["processing_seconds"], r["audio_seconds"])
                                                      for r in rows) / len(rows)),
    })
print("Tier 1 — ASR")
display(pd.DataFrame(asr_rows))

# --- Tier 2 + 3: parser and end-to-end ---
parser_rows = []
for (llm_model, source), group in pd.DataFrame(parser_results).groupby(["llm_model", "source"]):
    rows = group.to_dict("records")
    expected = [by_id[r["id"]]["expected"] for r in rows]
    drafts = [r["draft"] or {} for r in rows]
    successes = [is_task_success(r["draft"], exp) for r, exp in zip(rows, expected)]
    success_rate = sum(successes) / len(successes)
    deadline_pairs = [(parse_deadline(d.get("deadline_at")), parse_deadline(e["deadline_at"]))
                      for d, e in zip(drafts, expected)]
    with_deadline = [(predicted, true) for predicted, true in deadline_pairs if predicted and true]
    parser_rows.append({
        "llm_model": llm_model, "source": source, "n": len(rows),
        "schema_valid": round(sum(r["draft"] is not None for r in rows) / len(rows), 3),
        "category_acc": safe_metric(accuracy, [d.get("category") for d in drafts], [e["category"] for e in expected]),
        "category_macroF1": safe_metric(macro_f1, [d.get("category") for d in drafts], [e["category"] for e in expected]),
        "priority_acc": safe_metric(accuracy, [d.get("priority") for d in drafts], [e["priority"] for e in expected]),
        "deadline_day_acc": round(sum(same_deadline_day(d.get("deadline_at"), e["deadline_at"])
                                      for d, e in zip(drafts, expected)) / len(rows), 3),
        "deadline_MAE_h": safe_metric(lambda: mean_absolute_error_hours(
            [predicted for predicted, _ in with_deadline],
            [true for _, true in with_deadline])) if with_deadline else None,
        "deadline_invalid": sum(is_invalid_deadline(d.get("deadline_at")) for d in drafts),
        "title_tokenF1": safe_metric(lambda: sum(token_f1(d.get("title", ""), e["title"])
                                                 for d, e in zip(drafts, expected)) / len(rows)),
        "TASK_SUCCESS": round(success_rate, 3),
        "TSR_CI95": safe_metric(lambda: confidence_interval_95(success_rate, len(rows))[1] - success_rate),
        "latency_s (this machine)": round(group["processing_seconds"].mean(), 2),
    })
parser_table = pd.DataFrame(parser_rows)
print("Tier 2/3 — Parser & end-to-end   (TSR_CI95 = ± half-width)")
display(parser_table)

In [ ]:
# Δ_ASR: how much task success is lost because of ASR mistakes (audio examples only)
audio_ids = {e["id"] for e in audio_examples}
delta_rows = []
for llm_model in LLM_MODELS:
    rows = [r for r in parser_results if r["llm_model"] == llm_model and r["id"] in audio_ids]
    gold = [is_task_success(r["draft"], by_id[r["id"]]["expected"]) for r in rows if r["source"] == "gold"]
    for asr_model in ASR_MODELS:
        asr = [is_task_success(r["draft"], by_id[r["id"]]["expected"]) for r in rows if r["source"] == f"asr:{asr_model}"]
        if gold and asr:
            delta_rows.append({"llm_model": llm_model, "asr_model": asr_model,
                               "TSR_gold": sum(gold) / len(gold), "TSR_asr": sum(asr) / len(asr),
                               "Δ_ASR": sum(gold) / len(gold) - sum(asr) / len(asr)})
print("Δ_ASR = TSR(gold transcript) − TSR(ASR transcript)")
display(pd.DataFrame(delta_rows))

# Task success sliced by tag (gold transcripts) — shows WHERE each model fails
tag_rows = []
for r in parser_results:
    if r["source"] != "gold":
        continue
    for tag in by_id[r["id"]]["tags"]:
        tag_rows.append({"llm_model": r["llm_model"], "tag": tag,
                         "success": is_task_success(r["draft"], by_id[r["id"]]["expected"])})
print("Task success by tag (gold transcripts)")
display(pd.DataFrame(tag_rows).pivot_table(index="tag", columns="llm_model", values="success", aggfunc="mean").round(2))

In [ ]:
# Inspect failures one by one — reading errors teaches more than any average
failures = [
    {"llm": r["llm_model"], "source": r["source"], "id": r["id"], "transcript": r["transcript"],
     "pred_deadline": (r["draft"] or {}).get("deadline_at"), "exp_deadline": by_id[r["id"]]["expected"]["deadline_at"],
     "pred_cat": (r["draft"] or {}).get("category"), "exp_cat": by_id[r["id"]]["expected"]["category"],
     "pred_title": (r["draft"] or {}).get("title"), "error": r["error"]}
    for r in parser_results if not is_task_success(r["draft"], by_id[r["id"]]["expected"])
]
pd.set_option("display.max_colwidth", 60)
display(pd.DataFrame(failures))